In [1]:
from __future__ import annotations

import importlib.util
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from dotenv import load_dotenv

from fraud_intelligence.explainability.shap_explainer import (
    get_raw_feature_names,
    transform_features_for_explanation,
    validate_feature_frame,
)
from fraud_intelligence.models.model_loading import (
    load_frozen_xgboost_components,
)

NOTEBOOK_WORKING_DIRECTORY = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        candidate_directory
        for candidate_directory in (
            NOTEBOOK_WORKING_DIRECTORY,
            *NOTEBOOK_WORKING_DIRECTORY.parents,
        )
        if (
            (candidate_directory / "configs").is_dir()
            and (candidate_directory / "scripts").is_dir()
            and (candidate_directory / "src").is_dir()
        )
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the repository root. Expected an ancestor folder "
        "containing configs/, scripts/, and src/."
    )

os.chdir(PROJECT_ROOT)
load_dotenv(dotenv_path=PROJECT_ROOT / ".env")

FIGURE_DIRECTORY = PROJECT_ROOT / "reports" / "figures"
TABLE_DIRECTORY = PROJECT_ROOT / "reports" / "tables"

FIGURE_DIRECTORY.mkdir(parents=True, exist_ok=True)
TABLE_DIRECTORY.mkdir(parents=True, exist_ok=True)

MODELLING_CONFIG_PATH = PROJECT_ROOT / "configs" / "modeling.yaml"
PHASE5_HOLDOUT_SCRIPT_PATH = (
    PROJECT_ROOT / "scripts" / "evaluate_phase5_final_holdout.py"
)

with MODELLING_CONFIG_PATH.open(encoding="utf-8") as config_file:
    modelling_config = yaml.safe_load(config_file)

phase5_spec = importlib.util.spec_from_file_location(
    "phase5_final_holdout",
    PHASE5_HOLDOUT_SCRIPT_PATH,
)

if phase5_spec is None or phase5_spec.loader is None:
    raise ImportError(
        "Could not load scripts/evaluate_phase5_final_holdout.py."
    )

phase5_module = importlib.util.module_from_spec(phase5_spec)
phase5_spec.loader.exec_module(phase5_module)

if not hasattr(phase5_module, "load_final_test_data"):
    raise AttributeError(
        "The Phase 5 holdout script does not expose load_final_test_data()."
    )

print("Loading frozen Phase 5 champion model...")
frozen_components = load_frozen_xgboost_components()

print("Loading final holdout data...")
X_test, y_test, test_timestamps = phase5_module.load_final_test_data(
    modelling_config
)

raw_feature_names = get_raw_feature_names(
    frozen_components.preprocessor
)

X_test = validate_feature_frame(
    feature_frame=X_test,
    expected_input_features=raw_feature_names,
)

if len(X_test) != len(y_test):
    raise ValueError(
        "Final holdout features and labels have different lengths."
    )

if len(X_test) == 0:
    raise ValueError("The final holdout dataset is empty.")

configured_review_capacity = int(
    modelling_config["evaluation"]["review_capacity"]
)

review_capacity = min(configured_review_capacity, len(X_test))

print("Transforming and scoring final holdout transactions...")
X_test_transformed, _ = transform_features_for_explanation(
    preprocessor=frozen_components.preprocessor,
    raw_feature_frame=X_test,
)

raw_classifier_probabilities = (
    frozen_components.classifier.predict_proba(
        X_test_transformed
    )[:, 1]
)

if not np.isfinite(raw_classifier_probabilities).all():
    raise ValueError("The classifier returned non-finite probabilities.")

if not (
    (raw_classifier_probabilities >= 0.0)
    & (raw_classifier_probabilities <= 1.0)
).all():
    raise ValueError(
        "The classifier returned probabilities outside [0, 1]."
    )

ranked_holdout_positions = np.argsort(
    -raw_classifier_probabilities,
    kind="stable",
)

reviewed_mask = np.zeros(len(X_test), dtype=bool)
reviewed_mask[ranked_holdout_positions[:review_capacity]] = True

labels = y_test.to_numpy(dtype=int)
timestamps = np.asarray(test_timestamps)

if len(timestamps) != len(X_test):
    raise ValueError(
        "Final holdout timestamps and feature rows have different lengths."
    )

if "TransactionAmt" in X_test.columns:
    transaction_amounts = pd.to_numeric(
        X_test["TransactionAmt"],
        errors="coerce",
    ).to_numpy()
else:
    transaction_amounts = np.full(len(X_test), np.nan)

error_category = np.select(
    [
        (labels == 1) & reviewed_mask,
        (labels == 1) & ~reviewed_mask,
        (labels == 0) & reviewed_mask,
        (labels == 0) & ~reviewed_mask,
    ],
    [
        "captured_fraud_reviewed",
        "missed_fraud_not_reviewed",
        "unnecessary_review_legitimate",
        "correctly_low_risk_legitimate",
    ],
    default="unclassified",
)

holdout_analysis = pd.DataFrame(
    {
        "holdout_row_position": np.arange(len(X_test)),
        "transaction_timestamp": timestamps,
        "true_label": labels,
        "raw_classifier_probability": raw_classifier_probabilities,
        "reviewed_at_capacity": reviewed_mask,
        "error_category": error_category,
        "transaction_amount": transaction_amounts,
    }
)

category_order = [
    "captured_fraud_reviewed",
    "missed_fraud_not_reviewed",
    "unnecessary_review_legitimate",
    "correctly_low_risk_legitimate",
]

holdout_analysis["error_category"] = pd.Categorical(
    holdout_analysis["error_category"],
    categories=category_order,
    ordered=True,
)

error_summary = (
    holdout_analysis.groupby(
        "error_category",
        observed=False,
    )
    .agg(
        transaction_count=("true_label", "size"),
        fraud_count=("true_label", "sum"),
        mean_raw_score=("raw_classifier_probability", "mean"),
        median_raw_score=("raw_classifier_probability", "median"),
        minimum_raw_score=("raw_classifier_probability", "min"),
        maximum_raw_score=("raw_classifier_probability", "max"),
        mean_transaction_amount=("transaction_amount", "mean"),
        median_transaction_amount=("transaction_amount", "median"),
    )
    .reset_index()
)

error_summary["transaction_share"] = (
    error_summary["transaction_count"] / len(holdout_analysis)
)

error_summary.to_csv(
    TABLE_DIRECTORY / "phase7b_error_category_summary.csv",
    index=False,
)

policy_summary = pd.DataFrame(
    [
        {
            "model_name": "xgboost",
            "model_version": frozen_components.model_version,
            "mlflow_run_id": frozen_components.mlflow_run_id,
            "review_capacity": review_capacity,
            "holdout_transaction_count": len(holdout_analysis),
            "reviewed_transaction_count": int(reviewed_mask.sum()),
            "total_fraud_count": int(labels.sum()),
            "captured_fraud_count": int(
                ((labels == 1) & reviewed_mask).sum()
            ),
            "missed_fraud_count": int(
                ((labels == 1) & ~reviewed_mask).sum()
            ),
            "unnecessary_review_count": int(
                ((labels == 0) & reviewed_mask).sum()
            ),
            "fraud_capture_rate": float(
                ((labels == 1) & reviewed_mask).sum() / labels.sum()
            ),
            "review_precision": float(
                ((labels == 1) & reviewed_mask).sum() / reviewed_mask.sum()
            ),
            "policy_basis": (
                "Top-k ranking at the Phase 5 locked review capacity. "
                "Sigmoid calibration is monotonic, so it preserves the "
                "classifier score ranking used for review prioritisation."
            ),
        }
    ]
)

policy_summary.to_csv(
    TABLE_DIRECTORY / "phase7b_policy_error_summary.csv",
    index=False,
)

score_distribution_summary = (
    holdout_analysis.assign(
        label_name=np.where(
            holdout_analysis["true_label"] == 1,
            "fraud",
            "legitimate",
        )
    )
    .groupby("label_name", observed=True)
    .agg(
        transaction_count=("true_label", "size"),
        mean_raw_score=("raw_classifier_probability", "mean"),
        median_raw_score=("raw_classifier_probability", "median"),
        p05_raw_score=(
            "raw_classifier_probability",
            lambda values: values.quantile(0.05),
        ),
        p25_raw_score=(
            "raw_classifier_probability",
            lambda values: values.quantile(0.25),
        ),
        p75_raw_score=(
            "raw_classifier_probability",
            lambda values: values.quantile(0.75),
        ),
        p95_raw_score=(
            "raw_classifier_probability",
            lambda values: values.quantile(0.95),
        ),
    )
    .reset_index()
)

score_distribution_summary.to_csv(
    TABLE_DIRECTORY / "phase7b_score_distribution_summary.csv",
    index=False,
)

score_decile_labels = [
    f"score_decile_{decile}"
    for decile in range(1, 11)
]

holdout_analysis["score_decile"] = pd.qcut(
    holdout_analysis["raw_classifier_probability"].rank(
        method="first"
    ),
    q=10,
    labels=score_decile_labels,
)

score_decile_summary = (
    holdout_analysis.groupby(
        "score_decile",
        observed=False,
    )
    .agg(
        transaction_count=("true_label", "size"),
        fraud_count=("true_label", "sum"),
        fraud_rate=("true_label", "mean"),
        reviewed_count=("reviewed_at_capacity", "sum"),
        captured_fraud_count=(
            "true_label",
            lambda values: int(
                values[
                    holdout_analysis.loc[
                        values.index,
                        "reviewed_at_capacity",
                    ]
                ].sum()
            ),
        ),
        mean_raw_score=("raw_classifier_probability", "mean"),
        minimum_raw_score=("raw_classifier_probability", "min"),
        maximum_raw_score=("raw_classifier_probability", "max"),
    )
    .reset_index()
)

score_decile_summary.to_csv(
    TABLE_DIRECTORY / "phase7b_score_decile_summary.csv",
    index=False,
)

holdout_analysis["chronological_decile"] = pd.qcut(
    holdout_analysis["transaction_timestamp"].rank(
        method="first"
    ),
    q=10,
    labels=[
        f"time_period_{period}"
        for period in range(1, 11)
    ],
)

chronological_summary = (
    holdout_analysis.groupby(
        "chronological_decile",
        observed=False,
    )
    .agg(
        transaction_count=("true_label", "size"),
        fraud_count=("true_label", "sum"),
        fraud_rate=("true_label", "mean"),
        reviewed_count=("reviewed_at_capacity", "sum"),
        captured_fraud_count=(
            "true_label",
            lambda values: int(
                values[
                    holdout_analysis.loc[
                        values.index,
                        "reviewed_at_capacity",
                    ]
                ].sum()
            ),
        ),
        missed_fraud_count=(
            "error_category",
            lambda values: int(
                (values == "missed_fraud_not_reviewed").sum()
            ),
        ),
        unnecessary_review_count=(
            "error_category",
            lambda values: int(
                (values == "unnecessary_review_legitimate").sum()
            ),
        ),
        mean_raw_score=("raw_classifier_probability", "mean"),
        start_timestamp=("transaction_timestamp", "min"),
        end_timestamp=("transaction_timestamp", "max"),
    )
    .reset_index()
)

chronological_summary["fraud_capture_rate"] = (
    chronological_summary["captured_fraud_count"]
    / chronological_summary["fraud_count"].replace(0, np.nan)
)

chronological_summary.to_csv(
    TABLE_DIRECTORY / "phase7b_chronological_error_summary.csv",
    index=False,
)

high_priority_error_examples = (
    holdout_analysis.loc[
        holdout_analysis["error_category"].isin(
            [
                "missed_fraud_not_reviewed",
                "unnecessary_review_legitimate",
            ]
        )
    ]
    .sort_values(
        ["error_category", "raw_classifier_probability"],
        ascending=[True, False],
    )
    .groupby(
        "error_category",
        observed=True,
    )
    .head(20)
    .copy()
)

high_priority_error_examples.to_csv(
    TABLE_DIRECTORY / "phase7b_representative_error_cases.csv",
    index=False,
)

plt.figure(figsize=(10, 6))

for label_value, label_name, colour in [
    (0, "Legitimate", "#2563EB"),
    (1, "Fraud", "#DC2626"),
]:
    plt.hist(
        holdout_analysis.loc[
            holdout_analysis["true_label"] == label_value,
            "raw_classifier_probability",
        ],
        bins=50,
        alpha=0.55,
        label=label_name,
        density=True,
        color=colour,
    )

review_boundary_score = raw_classifier_probabilities[
    ranked_holdout_positions[review_capacity - 1]
]

plt.axvline(
    review_boundary_score,
    color="black",
    linestyle="--",
    linewidth=1.2,
    label=f"Top-{review_capacity:,} review boundary",
)

plt.xlabel("Raw frozen-XGBoost probability")
plt.ylabel("Density")
plt.title(
    "Phase 7B: Holdout Score Distributions by True Label"
)
plt.legend()
plt.tight_layout()
plt.savefig(
    FIGURE_DIRECTORY / "phase7b_score_distribution_by_label.png",
    dpi=200,
    bbox_inches="tight",
)
plt.close()

plot_error_summary = error_summary.copy()

plt.figure(figsize=(11, 6))
plt.bar(
    plot_error_summary["error_category"],
    plot_error_summary["transaction_count"],
    color=["#16A34A", "#DC2626", "#F59E0B", "#2563EB"],
)
plt.xticks(rotation=20, ha="right")
plt.ylabel("Holdout transactions")
plt.title(
    f"Phase 7B: Error Categories at Top-{review_capacity:,} Review Capacity"
)
plt.tight_layout()
plt.savefig(
    FIGURE_DIRECTORY / "phase7b_error_category_counts.png",
    dpi=200,
    bbox_inches="tight",
)
plt.close()

plt.figure(figsize=(11, 6))
plt.plot(
    chronological_summary["chronological_decile"].astype(str),
    chronological_summary["fraud_capture_rate"],
    marker="o",
    linewidth=2,
    color="#16A34A",
    label="Fraud capture rate",
)
plt.plot(
    chronological_summary["chronological_decile"].astype(str),
    chronological_summary["fraud_rate"],
    marker="o",
    linewidth=2,
    color="#DC2626",
    label="Observed fraud rate",
)
plt.ylim(bottom=0)
plt.xlabel("Chronological final-holdout period")
plt.ylabel("Rate")
plt.title(
    "Phase 7B: Fraud Rate and Capture Rate Across Holdout Time"
)
plt.legend()
plt.tight_layout()
plt.savefig(
    FIGURE_DIRECTORY / "phase7b_chronological_error_variation.png",
    dpi=200,
    bbox_inches="tight",
)
plt.close()

print("\n=== PHASE 7B ERROR ANALYSIS COMPLETE ===")
print(f"Champion model: {frozen_components.model_version}")
print(f"MLflow run ID: {frozen_components.mlflow_run_id}")
print(f"Review capacity: {review_capacity:,}")
print(
    "Captured fraud: "
    f"{int(policy_summary.loc[0, 'captured_fraud_count']):,} / "
    f"{int(policy_summary.loc[0, 'total_fraud_count']):,}"
)
print(
    "Missed fraud: "
    f"{int(policy_summary.loc[0, 'missed_fraud_count']):,}"
)
print(
    "Unnecessary reviews: "
    f"{int(policy_summary.loc[0, 'unnecessary_review_count']):,}"
)
print(
    "Fraud capture rate: "
    f"{float(policy_summary.loc[0, 'fraud_capture_rate']):.6f}"
)
print(
    "Review precision: "
    f"{float(policy_summary.loc[0, 'review_precision']):.6f}"
)

print("\nError-category summary:")
display(error_summary)

print("\nChronological error summary:")
display(chronological_summary)

print("\nSaved figures:")
for figure_path in sorted(FIGURE_DIRECTORY.glob("phase7b_*.png")):
    print(figure_path.relative_to(PROJECT_ROOT))

print("\nSaved tables:")
for table_path in sorted(TABLE_DIRECTORY.glob("phase7b_*.csv")):
    print(table_path.relative_to(PROJECT_ROOT))

Loading frozen Phase 5 champion model...


Loading final holdout data...
Transforming and scoring final holdout transactions...

=== PHASE 7B ERROR ANALYSIS COMPLETE ===
Champion model: xgboost-v1.0.0
MLflow run ID: a1f99cff34cf44728d3b6c9499ffeee3
Review capacity: 1,000
Captured fraud: 870 / 3,083
Missed fraud: 2,213
Unnecessary reviews: 130
Fraud capture rate: 0.282193
Review precision: 0.870000

Error-category summary:


,error_category,transaction_count,fraud_count,mean_raw_score,median_raw_score,minimum_raw_score,maximum_raw_score,mean_transaction_amount,median_transaction_amount,transaction_share
0,captured_fraud_reviewed,870,870,0.989410,0.993778,0.961143,0.999690,87.248201,40.700,0.009822
1,missed_fraud_not_reviewed,2213,2213,0.544224,0.560300,0.004672,0.960998,177.904467,77.000,0.024983
2,unnecessary_review_legitimate,130,0,0.982973,0.983874,0.961673,0.999426,61.273892,30.494,0.001468
3,correctly_low_risk_legitimate,85368,0,0.182044,0.119020,0.000211,0.960056,136.716100,68.950,0.963728



Chronological error summary:


,chronological_decile,transaction_count,fraud_count,fraud_rate,reviewed_count,captured_fraud_count,missed_fraud_count,unnecessary_review_count,mean_raw_score,start_timestamp,end_timestamp,fraud_capture_rate
0,time_period_1,8859,271,0.030590,98,94,177,4,0.181405,13151880,13374194,0.346863
1,time_period_2,8858,258,0.029126,76,71,187,5,0.203182,13374202,13639874,0.275194
2,time_period_3,8858,269,0.030368,75,65,204,10,0.196388,13639884,13902289,0.241636
3,time_period_4,8858,303,0.034206,163,126,177,37,0.221612,13902294,14159459,0.415842
4,time_period_5,8858,249,0.028110,46,36,213,10,0.190969,14159491,14417365,0.144578
5,time_period_6,8858,347,0.039174,81,77,270,4,0.205024,14417374,14672633,0.221902
6,time_period_7,8857,285,0.032178,85,73,212,12,0.178980,14672642,14939369,0.256140
7,time_period_8,8859,354,0.039959,133,118,236,15,0.207885,14939471,15207450,0.333333
8,time_period_9,8858,390,0.044028,135,125,265,10,0.210364,15207464,15533801,0.320513
9,time_period_10,8858,357,0.040303,108,85,272,23,0.206161,15533993,15811131,0.238095



Saved figures:
reports\figures\phase7b_chronological_error_variation.png
reports\figures\phase7b_error_category_counts.png
reports\figures\phase7b_score_distribution_by_label.png

Saved tables:
reports\tables\phase7b_chronological_error_summary.csv
reports\tables\phase7b_error_category_summary.csv
reports\tables\phase7b_policy_error_summary.csv
reports\tables\phase7b_representative_error_cases.csv
reports\tables\phase7b_score_decile_summary.csv
reports\tables\phase7b_score_distribution_summary.csv


In [ ]:
from pathlib import Path

import pandas as pd

REPORT_DIRECTORY = PROJECT_ROOT / "reports" / "evaluation"
REPORT_DIRECTORY.mkdir(parents=True, exist_ok=True)

error_summary = pd.read_csv(
    TABLE_DIRECTORY / "phase7b_error_category_summary.csv"
)
policy_summary = pd.read_csv(
    TABLE_DIRECTORY / "phase7b_policy_error_summary.csv"
)
score_distribution_summary = pd.read_csv(
    TABLE_DIRECTORY / "phase7b_score_distribution_summary.csv"
)
score_decile_summary = pd.read_csv(
    TABLE_DIRECTORY / "phase7b_score_decile_summary.csv"
)
chronological_summary = pd.read_csv(
    TABLE_DIRECTORY / "phase7b_chronological_error_summary.csv"
)
representative_error_cases = pd.read_csv(
    TABLE_DIRECTORY / "phase7b_representative_error_cases.csv"
)

policy = policy_summary.iloc[0].to_dict()


def markdown_table(
    table: pd.DataFrame,
    columns: list[str],
    decimal_columns: list[str] | None = None,
) -> str:
    """Create a Markdown table without optional dependencies."""
    decimal_columns = decimal_columns or []

    display_table = table.loc[:, columns].copy()

    for column in decimal_columns:
        if column in display_table.columns:
            display_table[column] = display_table[column].map(
                lambda value: f"{float(value):.6f}"
            )

    display_table = display_table.fillna("")

    for column in display_table.columns:
        display_table[column] = (
            display_table[column]
            .astype(str)
            .str.replace("|", "\\|", regex=False)
        )

    header = "| " + " | ".join(display_table.columns) + " |"
    separator = "| " + " | ".join(
        ["---"] * len(display_table.columns)
    ) + " |"

    rows = [
        "| " + " | ".join(row) + " |"
        for row in display_table.astype(str).values.tolist()
    ]

    return "\n".join([header, separator, *rows])


error_summary_display = error_summary.copy()
score_distribution_display = score_distribution_summary.copy()
score_decile_display = score_decile_summary.copy()
chronological_display = chronological_summary.copy()

for table in [
    error_summary_display,
    score_distribution_display,
    score_decile_display,
    chronological_display,
]:
    for column in table.columns:
        if "score" in column or "rate" in column or "share" in column:
            if pd.api.types.is_numeric_dtype(table[column]):
                table[column] = table[column].map(
                    lambda value: (
                        f"{float(value):.6f}"
                        if pd.notna(value)
                        else ""
                    )
                )

representative_error_display = representative_error_cases.copy()

for column in [
    "raw_classifier_probability",
    "transaction_amount",
]:
    if column in representative_error_display.columns:
        representative_error_display[column] = (
            representative_error_display[column].map(
                lambda value: (
                    f"{float(value):.6f}"
                    if pd.notna(value)
                    else ""
                )
            )
        )

missed_fraud_row = error_summary.loc[
    error_summary["error_category"] == "missed_fraud_not_reviewed"
].iloc[0]

unnecessary_review_row = error_summary.loc[
    error_summary["error_category"] == "unnecessary_review_legitimate"
].iloc[0]

captured_fraud_row = error_summary.loc[
    error_summary["error_category"] == "captured_fraud_reviewed"
].iloc[0]

lowest_period_capture = chronological_summary.loc[
    chronological_summary["fraud_capture_rate"].idxmin()
]

highest_period_capture = chronological_summary.loc[
    chronological_summary["fraud_capture_rate"].idxmax()
]

report_content = f"""# Phase 7B — Error Analysis Report

## Purpose

This report analyses final-holdout ranking and review errors for the frozen Phase 5
XGBoost champion model. It focuses on fraud cases missed outside the constrained
review cohort, legitimate transactions unnecessarily selected for review, score
overlap between fraud and legitimate transactions, and variation in results across
chronological holdout periods.

This is a public IEEE-CIS benchmark analysis under documented policy assumptions.
It must not be represented as a real financial-institution outcome, realised
savings estimate, or causal assessment of fraud behaviour.

## Locked Evaluation Scope

| Item | Value |
| --- | --- |
| Model name | `{policy["model_name"]}` |
| Model version | `{policy["model_version"]}` |
| MLflow training run | `{policy["mlflow_run_id"]}` |
| Final-holdout transaction count | `{int(policy["holdout_transaction_count"]):,}` |
| Locked review capacity | `{int(policy["review_capacity"]):,}` |
| Selected review count | `{int(policy["reviewed_transaction_count"]):,}` |
| Review-cohort selection basis | Top-k model-score ranking at fixed capacity |
| Calibration selected in Phase 5 | Sigmoid / Platt scaling |

The analysis loads the frozen final model and the untouched chronological final
holdout. No model fitting, hyperparameter tuning, calibration fitting, threshold
selection, or policy optimisation occurred in this Phase 7B notebook.

The review cohort is reconstructed as the highest-ranked
`{int(policy["review_capacity"]):,}` holdout transactions. Sigmoid calibration is
monotonic, so it preserves the classifier ranking used to construct this top-k
review cohort.

## Review Outcome Summary

At the locked review capacity, the model selected
`{int(policy["reviewed_transaction_count"]):,}` transactions for review. It captured
`{int(policy["captured_fraud_count"]):,}` of
`{int(policy["total_fraud_count"]):,}` fraud-labelled transactions, producing a fraud
capture rate of `{float(policy["fraud_capture_rate"]):.6f}` and review precision of
`{float(policy["review_precision"]):.6f}`.

- Captured fraud reviewed: `{int(policy["captured_fraud_count"]):,}`
- Missed fraud not reviewed: `{int(policy["missed_fraud_count"]):,}`
- Unnecessary legitimate reviews: `{int(policy["unnecessary_review_count"]):,}`
- Correctly low-risk legitimate transactions:
  `{int(policy["holdout_transaction_count"]) - int(policy["captured_fraud_count"]) - int(policy["missed_fraud_count"]) - int(policy["unnecessary_review_count"]):,}`

![Error category counts](../figures/phase7b_error_category_counts.png)

{markdown_table(
    error_summary_display,
    columns=[
        "error_category",
        "transaction_count",
        "fraud_count",
        "transaction_share",
        "mean_raw_score",
        "median_raw_score",
        "mean_transaction_amount",
        "median_transaction_amount",
    ],
)}

## Missed Fraud

The final holdout contains
`{int(missed_fraud_row["transaction_count"]):,}` missed fraud cases outside the
top-`{int(policy["review_capacity"]):,}` review cohort. These cases have a mean raw
classifier score of `{float(missed_fraud_row["mean_raw_score"]):.6f}` and a median
raw classifier score of `{float(missed_fraud_row["median_raw_score"]):.6f}`.

This result does not mean that the missed transactions were safe or non-fraudulent.
It shows that, under the fixed review-capacity constraint, their model scores did
not rank highly enough to enter the limited review cohort. Phase 7C will compare
capacity and decision-policy alternatives; this report only documents the error
population under the locked evaluation policy.

## Unnecessary Reviews

The model selected
`{int(unnecessary_review_row["transaction_count"]):,}` legitimate transactions for
review. These cases are false-positive review burden at the fixed capacity, not
evidence that the transactions were fraudulent.

Their mean raw classifier score was
`{float(unnecessary_review_row["mean_raw_score"]):.6f}`, compared with
`{float(captured_fraud_row["mean_raw_score"]):.6f}` for captured fraud. This score
overlap is the practical ranking trade-off: some legitimate transactions have
patterns that the frozen model associates with higher fraud risk, while some fraud
transactions receive comparatively low scores.

## Score Distributions

![Score distributions by true label](../figures/phase7b_score_distribution_by_label.png)

{markdown_table(
    score_distribution_display,
    columns=[
        "label_name",
        "transaction_count",
        "mean_raw_score",
        "median_raw_score",
        "p05_raw_score",
        "p25_raw_score",
        "p75_raw_score",
        "p95_raw_score",
    ],
)}

The score distributions should be interpreted as ranking evidence. Separation
between fraud and legitimate score distributions supports prioritisation, while
their overlap explains why both missed fraud and unnecessary reviews remain under
limited investigation capacity.

## Score-Decile Analysis

{markdown_table(
    score_decile_display,
    columns=[
        "score_decile",
        "transaction_count",
        "fraud_count",
        "fraud_rate",
        "reviewed_count",
        "captured_fraud_count",
        "mean_raw_score",
    ],
)}

The highest score deciles contain the constrained review cohort. Lower score
deciles still contain fraud-labelled transactions, illustrating the residual-risk
trade-off created by limited review capacity rather than a claim that lower-scored
transactions are definitively legitimate.

## Chronological Variation

![Chronological error variation](../figures/phase7b_chronological_error_variation.png)

{markdown_table(
    chronological_display,
    columns=[
        "chronological_decile",
        "transaction_count",
        "fraud_count",
        "fraud_rate",
        "reviewed_count",
        "captured_fraud_count",
        "missed_fraud_count",
        "unnecessary_review_count",
        "fraud_capture_rate",
        "mean_raw_score",
    ],
)}

Across the ten chronological final-holdout periods, fraud capture rate ranged from
`{float(lowest_period_capture["fraud_capture_rate"]):.6f}` in
`{lowest_period_capture["chronological_decile"]}` to
`{float(highest_period_capture["fraud_capture_rate"]):.6f}` in
`{highest_period_capture["chronological_decile"]}`.

This temporal variation is descriptive benchmark evidence. It may reflect changing
transaction composition, fraud prevalence, feature distributions, or model-ranking
difficulty. It does not establish a causal explanation.

## Representative Error Cases

The following records are limited examples selected from the two principal error
populations. They support qualitative inspection but are not population-level
estimates.

{markdown_table(
    representative_error_display,
    columns=[
        "error_category",
        "holdout_row_position",
        "transaction_timestamp",
        "true_label",
        "raw_classifier_probability",
        "reviewed_at_capacity",
        "transaction_amount",
    ],
)}

## Limitations

- This analysis uses the public IEEE-CIS Fraud Detection benchmark and documented
  illustrative decision assumptions; it is not real banking or card-network data.
- A fixed top-k review capacity constrains fraud capture by design. Missed fraud
  cases are not necessarily low risk; they were simply outside the selected review
  cohort at the locked capacity.
- The analysis uses model score ranking to reconstruct the fixed-size review cohort.
  It does not re-optimise the threshold, capacity, calibration method, or action
  policy on the holdout.
- Raw classifier probabilities are shown for ranking/error-analysis context. The
  approved confidence-calibration method remains Phase 5 sigmoid / Platt scaling.
- Score distributions and chronological summaries identify associations and
  operational trade-offs, not causal drivers of fraud.
- The transaction-level error-case export is limited to representative records and
  does not replace the aggregate summaries used for portfolio conclusions.

## Output Inventory

```text
reports/figures/phase7b_error_category_counts.png
reports/figures/phase7b_score_distribution_by_label.png
reports/figures/phase7b_chronological_error_variation.png
reports/tables/phase7b_error_category_summary.csv
reports/tables/phase7b_policy_error_summary.csv
reports/tables/phase7b_score_distribution_summary.csv
reports/tables/phase7b_score_decile_summary.csv
reports/tables/phase7b_chronological_error_summary.csv
reports/tables/phase7b_representative_error_cases.csv
```
"""

report_path = REPORT_DIRECTORY / "error_analysis_report.md"
report_path.write_text(report_content, encoding="utf-8")

print("=== PHASE 7B REPORT CREATED ===")
print(f"Report: {report_path.relative_to(PROJECT_ROOT)}")
print(f"Report size: {report_path.stat().st_size:,} bytes")
print(
    "Missed fraud: "
    f"{int(policy['missed_fraud_count']):,}"
)
print(
    "Unnecessary reviews: "
    f"{int(policy['unnecessary_review_count']):,}"
)
print(
    "Chronological fraud-capture range: "
    f"{float(lowest_period_capture['fraud_capture_rate']):.6f} to "
    f"{float(highest_period_capture['fraud_capture_rate']):.6f}"
)